# AAPL — four priors projected onto one market-quote constraint set

Design and predictions pre-registered in `taskc/DECISIONS.md` section 26 (with amendments 26.8 and
26.9), all written before any arm result existed.

**Scope, stated first.** One date, one underlying, maturities 7–25 calendar days. This supports **no**
claim about term structure, regime stability, or any other date. It is a single cross-sectional fit.

| arm | prior |
|---|---|
| 1 | **Path DDPM** on AAPL returns — the frozen recipe (450 epochs, cosine lr decay, EMA) |
| 2 | **Heston** fitted to the same returns by method of moments — genuinely misspecified |
| 3 | **Student-t i.i.d.** returns, MLE — a parametric floor with fat tails, no vol clustering |
| 4 | **Black–Scholes** at historical vol — i.i.d. Gaussian, the trivial baseline |

Calibrated on the **longest** maturity (25 dte → step 17) and held out on the four **shorter** ones,
so every held-out instrument is an extrapolation in maturity, not an interpolation between calibrated
strikes. Vanilla constraints are **intervals** `bid ≤ E_Q[payoff] ≤ ask` (26.8): forcing the mid is
infeasible here and asserts a precision the quote denies.

In [ ]:
PINNED_COMMIT    = "8bd04ea771fd0debfa6403970e3e2412ddd45535"
NOTEBOOK_VERSION = "aapl-2026.09.25e"
EXPECT_TASKC     = "taskc-2026.09.25c"

%cd /content
%rm -rf ddpm_option_pricing
!git clone -q https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch -q --all
!git checkout -q {PINNED_COMMIT}
import subprocess
HEAD = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
print("pinned :", PINNED_COMMIT); print("HEAD   :", HEAD); print("notebook:", NOTEBOOK_VERSION)
assert HEAD.startswith(PINNED_COMMIT) or PINNED_COMMIT.startswith(HEAD), "checkout missed the pinned commit -- stop"
!git log --oneline -1

### Stage 0 — environment, full fp32 (TF32 off), Drive

In [ ]:
import os, sys, json, math, time
import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import taskc
from taskc.config import frozen
from taskc.data import PathStandardizer, make_loader
from taskc.ptheta import (make_schedule, build_model, train_ptheta_decay_ema,
                          save_checkpoint, load_checkpoint, sample_ptheta)
from taskc.aapl import (load_surface, spot_of, calls_at, parity_forward, carry_from_forward,
                        paths_from_returns, vanilla_columns, build_aapl, heldout_errors,
                        fit_heston_mom, sample_heston_returns, fit_student_t, sample_student_t,
                        fit_gaussian, sample_gaussian, price_exotics, solve_interval,
                        R_PINNED, CALIB_DTE, MAX_REL_SPREAD, aapl_exotics)

_nb_path = "notebooks/taskc_13_aapl_colab.ipynb"
assert taskc.__version__ == EXPECT_TASKC, f"stale: taskc {taskc.__version__} != {EXPECT_TASKC}"
assert NOTEBOOK_VERSION in open(_nb_path).read(), (
    f"{NOTEBOOK_VERSION} does not appear in this notebook at PINNED_COMMIT ({PINNED_COMMIT[:7]}). "
    f"Stale cached notebook, or the pin was not advanced with the version.")

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/ddpm_aapl"
except Exception:
    DRIVE = os.environ.get("DRY_OUT", "artifacts_taskc/aapl_local")
os.makedirs(DRIVE, exist_ok=True)
print("taskc:", taskc.__version__, "| device:", DEVICE, "| tf32:", torch.backends.cuda.matmul.allow_tf32)

### Stage 1 — data

The surface is committed (it cannot be regenerated). Returns are fetched with **pinned dates** and
cross-checked against the committed frozen history; a mismatch is reported, not absorbed.

In [ ]:
N_PATHS   = 200_000       # paths per arm
N_TRAIN   = None          # all available 21-day blocks (overlapping, stride 1)
H, DT     = 21, 1/252
START, END = "2015-01-02", "2026-04-21"     # yfinance end is exclusive -> through 2026-04-20
SPREAD_FLOORS = (0.00, 0.05, 0.10, 0.15, 0.25)   # 26.R.3
SKIP_DONE = True

RUN = frozen(artifact_dir=DRIVE, data_dim=H)
RES = os.path.join(DRIVE, "aapl.json")
res = json.load(open(RES)) if (SKIP_DONE and os.path.exists(RES)) else dict(
    notebook=NOTEBOOK_VERSION, pinned=PINNED_COMMIT, taskc=taskc.__version__, device=DEVICE,
    window=[START, END], n_paths=N_PATHS, r_pinned=R_PINNED, calib_dte=CALIB_DTE,
    max_rel_spread=MAX_REL_SPREAD, stages={}, timings={})
# 26.R.5: `notebook` was stamped once at file creation, so a file created by one
# version and filled in by a later one mis-reported its own provenance. Each stage
# now carries the version that produced IT, and the top level records both.
res["notebook_created_by"] = res.get("notebook_created_by", res.get("notebook", NOTEBOOK_VERSION))
res["notebook_last_run"] = NOTEBOOK_VERSION
def stamp(dct):
    dct["_notebook"] = NOTEBOOK_VERSION
    return dct
def save(): json.dump(res, open(RES,"w"), indent=1, default=float)

SURF = "data/aapl/aapl_option_chain_clean.csv"
d = load_surface(SURF); S0 = spot_of(d)
r_impl, F, nK = parity_forward(d, CALIB_DTE)
CARRY = carry_from_forward(F, S0, CALIB_DTE)
cal  = calls_at(d, CALIB_DTE)
held = pd.concat([calls_at(d, x) for x in sorted(set(d.dte) - {CALIB_DTE})], ignore_index=True)
print(f"surface as-of 2026-04-20 | S0 {S0:.4f} | calib dte {CALIB_DTE} -> step {int(cal.step.iloc[0])}")
print(f"parity: F {F:.4f} from {nK} strikes | implied r {r_impl*100:+.2f}% (NOT identified, unused) "
      f"| carry b {CARRY*100:+.3f}%/yr | discount r pinned {R_PINNED*100:.1f}%")
print(f"calibrated calls {len(cal)} (K {cal.strike.min():.0f}-{cal.strike.max():.0f}) | "
      f"held-out {len(held)} over dte {sorted(held.dte.unique())} -> steps {sorted(held.step.unique())}")

# returns: pinned fetch, frozen cross-check
froz = pd.read_csv("data/aapl/aapl_stock_history.csv", parse_dates=["Date"])
froz = froz[(froz.Date >= START) & (froz.Date <= "2026-04-20")]
rets_frozen = np.diff(np.log(froz["Close"].to_numpy()))
# prices_yf.py lives on branch v2_aapl, which this pinned checkout does not contain.
# The clone has every branch, so materialise exactly that file rather than merging
# branches or vendoring a copy that could drift from it.
try:
    import subprocess, importlib.util
    src = subprocess.run(["git", "show", "origin/v2_aapl:src_real/data/prices_yf.py"],
                         capture_output=True, text=True, check=True).stdout
    open("/tmp/prices_yf.py", "w").write(src)
    spec = importlib.util.spec_from_file_location("prices_yf", "/tmp/prices_yf.py")
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    load_prices_yfinance = mod.load_prices_yfinance
    px = load_prices_yfinance("AAPL", start=START, end=END, price_col="Close")
    rets_live = np.diff(np.log(px))
    n = min(len(rets_live), len(rets_frozen))
    dev = float(np.abs(rets_live[-n:] - rets_frozen[-n:]).max())
    live_ok = dev < 1e-8 and len(rets_live) == len(rets_frozen)
    print(f"yfinance fetch: {len(rets_live)} returns vs frozen {len(rets_frozen)}; max|diff| {dev:.2e} "
          f"-> {'MATCH' if live_ok else 'MISMATCH (vendor revision); using FROZEN'}")
except Exception as e:
    live_ok = False
    print(f"yfinance fetch unavailable ({type(e).__name__}: {e}); using the committed frozen history")
rets = rets_frozen
res["stages"]["data"] = dict(S0=S0, F=F, carry=CARRY, r_implied=r_impl, n_parity=nK,
                             n_calib=len(cal), n_heldout=len(held), n_returns=len(rets),
                             ann_vol=float(rets.std()*math.sqrt(252)), live_match=bool(live_ok))
stamp(res["stages"]["data"])
save()
print(f"returns {len(rets)} | ann vol {rets.std()*math.sqrt(252)*100:.2f}%")

### Stage 2 — fit the three parametric arms

In [ ]:
if "fits" not in res["stages"]:
    hes = fit_heston_mom(rets, dt=DT)
    st  = fit_student_t(rets)
    ga  = fit_gaussian(rets)
    res["stages"]["fits"] = dict(
        heston=dict(kappa=hes.kappa, theta=hes.theta, xi=hes.xi, rho=hes.rho, v0=hes.v0,
                    drift=hes.drift, feller=hes.feller, ann_vol=float(np.sqrt(hes.theta))),
        student_t=st, gaussian=ga)
    stamp(res["stages"]["fits"])
    save()
FIT = res["stages"]["fits"]
print("Heston MoM  :", {k: round(v,5) for k,v in FIT["heston"].items()})
print("  Feller", f"{FIT['heston']['feller']:+.4f}",
      "(violated -> full-truncation Euler; reported, not constrained away)" if FIT["heston"]["feller"]<0 else "")
print("Student-t   :", {k: round(v,6) for k,v in FIT["student_t"].items()})
print("Gaussian/BS :", {k: round(v,6) for k,v in FIT["gaussian"].items()},
      f"-> ann vol {FIT['gaussian']['scale']*math.sqrt(252)*100:.2f}%")

### Stage 3 — the learned prior

Overlapping 21-day blocks (stride 1), the frozen recipe, and taskc's **single global** standardizer
(not `src_real`'s per-dimension one) so it is the same object as everywhere else in the paper.

In [ ]:
CKPT = os.path.join(DRIVE, "ptheta_aapl.pt")
if "train" not in res["stages"]:
    t0 = time.time()
    blocks = np.lib.stride_tricks.sliding_window_view(rets, H)      # (N, 21), stride 1
    std = PathStandardizer.fit(blocks, S0=S0, r=CARRY, dt=DT)
    z = std.to_z(blocks).astype(np.float32)
    keep = np.abs(z).max(axis=1) <= RUN.z_cap
    print(f"blocks {blocks.shape}, {int((~keep).sum())} rejected by the cap -> {int(keep.sum())}")
    m = build_model(RUN).to(DEVICE); sched = make_schedule(RUN, device=DEVICE)
    ld = make_loader(z[keep], batch_size=RUN.batch_size, seed=RUN.init_seed)
    m = train_ptheta_decay_ema(m, ld, sched, RUN, device=DEVICE); m.eval()
    save_checkpoint(CKPT, m, std, RUN, extra=dict(world="aapl", n_blocks=int(keep.sum()),
                                                  train_seconds=time.time()-t0))
    res["stages"]["train"] = dict(n_blocks=int(keep.sum()), rejected=int((~keep).sum()),
                                  seconds=time.time()-t0)
    stamp(res["stages"]["train"]); res["timings"]["train"]=time.time()-t0; save()
model, std, ck_cfg, ck_extra = load_checkpoint(CKPT, device=DEVICE)
sched = make_schedule(RUN, device=DEVICE)
assert ck_cfg["epochs"] == 450 and ck_cfg["lr_decay"] and ck_cfg["ema"], f"not frozen recipe: {ck_extra}"
print("checkpoint:", ck_extra)

### Stage 4 — project all four arms onto the identical constraint set

41 C0 martingale columns (targets exactly 0, so equalities) + the calibrated vanilla columns as
**bid/ask intervals**.

In [ ]:
res["stages"].setdefault("arms", {})
if len(res["stages"]["arms"]) < 4:
    t0 = time.time()
    Y = {}
    DRAW = os.path.join(DRIVE, "ddpm_draw.npz")          # cache: the only expensive sample
    if not os.path.exists(DRAW):
        z = sample_ptheta(model, sched, n=N_PATHS, seed=7001, cfg=RUN,
                          device=DEVICE, verbose=False).z.astype(np.float32)
        np.savez_compressed(DRAW, z=z)
    Y["ddpm"] = std.to_Y(np.load(DRAW)["z"].astype(np.float64))
    Y["heston_mom"] = sample_heston_returns(
        __import__("config").Heston(S0=S0, v0=FIT["heston"]["v0"], drift=FIT["heston"]["drift"], r=0.0,
                                    kappa=FIT["heston"]["kappa"], theta=FIT["heston"]["theta"],
                                    xi=FIT["heston"]["xi"], rho=FIT["heston"]["rho"]),
        N_PATHS, H, seed=7002, dt=DT)
    Y["student_t"] = sample_student_t(FIT["student_t"], N_PATHS, H, seed=7003)
    Y["bs_gaussian"] = sample_gaussian(FIT["gaussian"], N_PATHS, H, seed=7004)

    out = res["stages"]["arms"]
    emp_max = float(np.abs(rets).max())
    for arm, Ya in Y.items():
        if arm in out:
            print(f"{arm:12s} cached"); continue
        # Stop only on a NUMERICALLY broken prior. An earlier version asserted
        # max|Y| < 1 and stopped the Student-t arm at 2.41, which is simply what a
        # df = 3.17 t produces over 4.2e6 draws (expected max 1.44). Heavy-tailed is
        # not the same as broken, so the tail is REPORTED, not enforced (26.10).
        bad = int((~np.isfinite(Ya)).sum())
        assert bad == 0, f"{arm}: {bad} non-finite returns"
        assert np.abs(Ya).max() < 10.0, (
            f"{arm}: max |daily log-return| {np.abs(Ya).max():.3g} -- a factor of e^10 in one day "
            f"is a broken generator, not a fat tail")
        tail = {f"gt_{t}": int((np.abs(Ya) > t).sum()) for t in (0.15, 0.25, 0.50, 1.00)}
        print(f"{arm:12s} tail: max|Y| {np.abs(Ya).max():.4f} (data {emp_max:.4f})  " +
              " ".join(f"|Y|>{t}: {tail['gt_'+str(t)]}" for t in (0.15, 0.25, 0.5, 1.0)))
        P = paths_from_returns(Ya, S0, CARRY, DT)
        assert np.isfinite(P.S).all(), f"{arm}: non-finite prices"
        cs = build_aapl(P, cal, R_PINNED)
        van = np.array([k == "vanilla" for k in cs.kinds])
        _, _, tgt, sp = vanilla_columns(P, cal, R_PINNED)
        lo, hi = cs.c.copy(), cs.c.copy()
        lo[van] = tgt - sp/2; hi[van] = tgt + sp/2          # bid .. ask
        w, beta, kl, ess, info = solve_interval(cs.G, lo, hi)
        he = heldout_errors(P, held, w, R_PINNED)
        ex = price_exotics(P, S0, w, R_PINNED)
        cal_err = heldout_errors(P, cal, w, R_PINNED)
        out[arm] = dict(
            ess=ess, kl=kl, max_violation=info["max_violation"], n_violated=info["n_violated"],
            n_at_bid=info["n_at_lower"], n_at_ask=info["n_at_upper"],
            calib=dict(med_abs_spreads=float(cal_err.err_spreads.abs().median()),
                       max_abs_spreads=float(cal_err.err_spreads.abs().max())),
            heldout=dict(med_abs_spreads=float(he.err_spreads.abs().median()),
                         max_abs_spreads=float(he.err_spreads.abs().max()),
                         by_dte={str(k): float(g.err_spreads.abs().median())
                                 for k, g in he.groupby("dte")},
                         # 26.R.3: the raw metric is dominated by 2-8 cent denominators
                         # in the OTM wing, so report it against a floor on the spread.
                         by_floor={f"{f:.2f}": dict(
                             n=int((he.spread >= f).sum()),
                             med=float(he.loc[he.spread >= f, "err_spreads"].abs().median()),
                             max=float(he.loc[he.spread >= f, "err_spreads"].abs().max()))
                             for f in SPREAD_FLOORS}),
            heldout_contracts=he.to_dict("records"),
            exotics=ex, tail=tail, max_abs_return=float(np.abs(Ya).max()))
        stamp(out[arm])
        print(f"{arm:12s} ESS {ess*100:6.2f}%  KL {kl:.4f}  viol {info['max_violation']:.1e}  "
              f"bid/ask bound {info['n_at_lower']}/{info['n_at_upper']}  "
              f"held-out |err| med {out[arm]['heldout']['med_abs_spreads']:5.2f} sp", flush=True)
        save()                                   # per arm, so a later failure costs nothing
    res["timings"]["arms"] = time.time()-t0; save()
print("done")

### Stage 5 — the table

In [ ]:
A = res["stages"]["arms"]
LAB = {"ddpm":"Path DDPM (learned)","heston_mom":"Heston MoM","student_t":"Student-t iid","bs_gaussian":"Black-Scholes"}
print("="*112); print("PROJECTION"); print("="*112)
print(f"{'arm':24s} {'ESS %':>8s} {'KL':>9s} {'max viol':>10s} {'at bid':>7s} {'at ask':>7s} "
      f"{'calib |err| med/max':>21s}")
for a in ["ddpm","heston_mom","student_t","bs_gaussian"]:
    v=A[a]; print(f"{LAB[a]:24s} {v['ess']*100:8.2f} {v['kl']:9.4f} {v['max_violation']:10.2e} "
                  f"{v['n_at_bid']:7d} {v['n_at_ask']:7d} "
                  f"{v['calib']['med_abs_spreads']:10.2f}/{v['calib']['max_abs_spreads']:9.2f}")
print("\n" + "="*112)
print("HELD-OUT vanillas, |model - mid| in BID-ASK SPREAD units  (maturity extrapolation: step 17 -> 5..12)")
print("="*112)
dtes = sorted({int(k) for a in A for k in A[a]["heldout"]["by_dte"]})
print(f"{'arm':24s} " + "".join(f"{'dte '+str(x):>9s}" for x in dtes) + f" {'median':>9s} {'max':>9s}")
for a in ["ddpm","heston_mom","student_t","bs_gaussian"]:
    v=A[a]["heldout"]
    print(f"{LAB[a]:24s} " + "".join(f"{v['by_dte'].get(str(x), float('nan')):9.2f}" for x in dtes)
          + f" {v['med_abs_spreads']:9.2f} {v['max_abs_spreads']:9.2f}")
print("\n" + "="*112); print("EXOTICS (discounted at the pinned r; struck off S0 = %.3f)" % res["stages"]["data"]["S0"])
print("="*112)
ks = list(A["ddpm"]["exotics"])
print(f"{'arm':24s} " + "".join(f"{k[:22]:>24s}" for k in ks))
for a in ["ddpm","heston_mom","student_t","bs_gaussian"]:
    print(f"{LAB[a]:24s} " + "".join(f"{A[a]['exotics'][k]['price']:16.4f}({A[a]['exotics'][k]['se']:6.4f})" for k in ks))
print()
for k in ks:
    p=[A[a]["exotics"][k]["price"] for a in A]
    print(f"  {k:26s} spread across arms {max(p)-min(p):8.4f}  = {100*(max(p)-min(p))/np.mean(p):6.2f}% of mean")
print("\n" + "="*112)
print("HELD-OUT median |err| in spread units vs a FLOOR on the spread (26.R.3)")
print("="*112)
fl = [f"{f:.2f}" for f in SPREAD_FLOORS]
print(f"{'arm':24s} " + "".join(f"{'floor '+f:>13s}" for f in fl))
print(f"{'contracts kept':24s} " + "".join(
    f"{A['ddpm']['heldout']['by_floor'][f]['n']:13d}" for f in fl))
for a in ["ddpm","heston_mom","student_t","bs_gaussian"]:
    bf = A[a]["heldout"]["by_floor"]
    print(f"{LAB[a]:24s} " + "".join(f"{bf[f]['med']:13.3f}" for f in fl))
print("\nThe raw column is dominated by 2-8 cent denominators in the OTM wing; quote the")
print("0.10 floor as the headline with the raw figure beside it (26.R.3).")

print("\n" + "="*112)
print("TAIL BEHAVIOUR of each prior vs the data it was fitted to (26.10)")
print("="*112)
print(f"{'arm':24s} {'max |daily ret|':>16s} " + "".join(f"{'|r|>'+t:>10s}" for t in ("0.15","0.25","0.5","1.0")))
for a in ["ddpm","heston_mom","student_t","bs_gaussian"]:
    v=A[a]
    print(f"{LAB[a]:24s} {v.get('max_abs_return',float('nan')):16.4f} " +
          "".join(f"{v.get('tail',{}).get('gt_'+t,0):10d}" for t in ("0.15","0.25","0.5","1.0")))
print(f"{'AAPL data (2,839 obs)':24s} {float(np.abs(rets).max()):16.4f} " +
      "".join(f"{int((np.abs(rets)>float(t)).sum()):10d}" for t in ("0.15","0.25","0.5","1.0")))
print("\nSCOPE: one date (2026-04-20), one underlying, 7-25 calendar days. No claim about term")
print("structure, regime stability, or any other date follows from this.")
print("timings (min):", {k: round(v/60,2) for k,v in res["timings"].items()})

Results in `aapl.json`. Outcomes go to DECISIONS.md section 26.R against the 26.6 predictions,
including any that are falsified.